# Linear Regression: Closed-Form Solution (Normal Equations)

This notebook derives and implements the closed-form solution for linear regression, and validates it against scikit-learn's `LinearRegression`. We start with a single synthetic predictor, then move to a single real predictor, and finally generalize to multiple real predictors using the scikit-learn diabetes dataset (see the companion notebook, `linear_regression_and_feature_selection.ipynb`, for the full dataset description and the evaluation/feature-selection work built on top of it).

### Simple Linear Regression with Closed-Form Solution (Normal Equations)

$$ w = \frac{\sum_{i=1}^n x_i y_i - \frac{1}{n}\left(\sum_{i=1}^n x_i\right)\left(\sum_{i=1}^n y_i\right)}{\sum_{i=1}^n x_i^2 - \frac{1}{n}\left(\sum_{i=1}^n x_i\right)^2} $$

$$ b = \frac{1}{n}\sum_{i=1}^n y_i - \frac{w}{n}\sum_{i=1}^n x_i $$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

# Data generation
x = np.random.uniform(0, 10, 100)
y = 5 + 2*x + np.random.normal(0, 1, 100)

# Compute coefficients (closed-form)
n = len(x)
w = ((x*y).sum() - (1./n)*x.sum()*y.sum()) / ((x*x).sum() - (1./n)*(x.sum()**2))
b = (1./n)*y.sum() - (w/n)*x.sum()
print("Modelo: y =", b, "+", w, "* x")

# Predictions and residuals
y_pred = w*x + b
r = y - y_pred

# ---- subplots ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Datos originales
axes[0].scatter(x, y)
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
axes[0].set_title('Datos')

# línea de regression
idx = np.argsort(x)
axes[1].scatter(x, y)
axes[1].plot(x[idx], y_pred[idx], color='red', linewidth=2)
axes[1].set_xlabel('x')
axes[1].set_ylabel('y')
axes[1].set_title('Modelo')

# Residuos
axes[2].scatter(y, r)
axes[2].axhline(0, color='red', linestyle='--', linewidth=2)
axes[2].set_xlabel('y')
axes[2].set_ylabel('Error')
axes[2].set_title('Residuos')

plt.tight_layout()
plt.show()


### Closed-form solution applied to a single feature of the diabetes dataset

The formula above only works for one predictor. Let's apply it to real data, using a single feature from the diabetes dataset (`bmi`) instead of synthetic data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.linear_model import LinearRegression

diabetes = datasets.load_diabetes()
features = diabetes.feature_names

# Select a single feature
feat_idx = features.index('bmi')
x = diabetes.data[:, feat_idx]
y = diabetes.target

# Compute coefficients (closed-form)
n = len(x)
w = ((x*y).sum() - (1./n)*x.sum()*y.sum()) / ((x*x).sum() - (1./n)*(x.sum()**2))
b = (1./n)*y.sum() - (w/n)*x.sum()
print("Modelo: y =", b, "+", w, "* x  (feature:", features[feat_idx], ")")

# Predictions and residuals
y_pred = w*x + b
r = y - y_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(x, y)
axes[0].set_xlabel(features[feat_idx])
axes[0].set_ylabel('y')
axes[0].set_title('Datos')

idx = np.argsort(x)
axes[1].scatter(x, y)
axes[1].plot(x[idx], y_pred[idx], color='red', linewidth=2)
axes[1].set_xlabel(features[feat_idx])
axes[1].set_ylabel('y')
axes[1].set_title('Modelo')

axes[2].scatter(y, r)
axes[2].axhline(0, color='red', linestyle='--', linewidth=2)
axes[2].set_xlabel('y')
axes[2].set_ylabel('Error')
axes[2].set_title('Residuos')

plt.tight_layout()
plt.show()

# Compare against sklearn
regr = LinearRegression()
regr.fit(x.reshape(-1, 1), y)
print("\nComparación con sklearn:")
print("  Cerrada:  w =", w, " b =", b)
print("  sklearn:  w =", regr.coef_[0], " b =", regr.intercept_)

### Closed-form solution generalized to multiple features (normal equation, matrix form)

The scalar formula only works for a single predictor. To fit all 10 features of the diabetes dataset at once, we need the matrix version of the normal equation:

$$ w = (X^TX)^{-1}X^Ty $$

where a column of ones is added to X to account for the intercept.

In [ ]:
import numpy as np
from sklearn import datasets
from sklearn.linear_model import LinearRegression

diabetes = datasets.load_diabetes()
X = diabetes.data
y = diabetes.target
features = diabetes.feature_names

# Add a column of ones to X to account for the intercept
X_b = np.hstack([np.ones((X.shape[0], 1)), X])

# Closed-form solution (normal equation, matrix form)
theta = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y
b_closed = theta[0]
w_closed = theta[1:]

print("Modelo (forma cerrada matricial):")
print("  Intercepto:", b_closed)
for name, coef in zip(features, w_closed):
    print(f"  {name}: {coef:.4f}")

# Compare against sklearn
regr = LinearRegression()
regr.fit(X, y)

print("\nModelo (sklearn):")
print("  Intercepto:", regr.intercept_)
for name, coef in zip(features, regr.coef_):
    print(f"  {name}: {coef:.4f}")

print("\nDiferencia máxima en coeficientes:", np.max(np.abs(w_closed - regr.coef_)))